# Parsing of scaling law results

This notebook has been written to run on the TU Ilmenau cluster with my specific setup.
A summary of all results is also stored in the directory `/research_e3/scaling_data`.

In [ ]:
from __future__ import annotations

import os
import re
import json
import numpy as np
from typing import Any
from collections import defaultdict
from tensorboard.backend.event_processing import event_accumulator

import torch_geometric

from optimetal_e3.evaluation import Evaluator
from optimetal_e3.data.loader import load_torch_data, create_dataloader

def init_empty_results() -> dict:
    """
    Initialize an empty 'results' structure for the scaling law study.
    """
    return {
        "lmax0": {},
        "lmax1": {},
        "lmax2": {},
        "lmax3": {},
    }

def extract_processed_dirs(results: dict) -> set[str]:
    """
    Recursively walk all leaf lists in results and collect their 'study_dir'.
    """
    processed = set()
    def _walk(obj: Any) -> None:
        if isinstance(obj, dict):
            for v in obj.values():
                _walk(v)
        elif isinstance(obj, list):
            for entry in obj:
                if "study_dir" in entry:
                    processed.add(entry["study_dir"])
    _walk(results)
    return processed

def load_tb_scalars(logdir: str) -> dict:
    """
    Load scalar values from tensorboard event files. This is useful
    when you want to look at training and validation loss curves.
    """
    ea = event_accumulator.EventAccumulator(
        logdir,
        size_guidance={event_accumulator.SCALARS: 0},
    )
    ea.Reload()
    tags = ea.Tags().get("scalars", [])
    tb_log = {}
    for tag in tags:
        events = ea.Scalars(tag)
        values = [e.value for e in events]
        tb_log[tag] = values
    return tb_log

def eval_model(
    best_model_path: str, 
    dataloader: torch_geometric.loader.DataLoader,
    device_index=0,
) -> dict:
    """
    Use the Evaluator class to gather metrics for a given model.
    """
    evaluator = Evaluator(
        best_model_path=best_model_path, 
        dataloader=dataloader, 
        device_index=device_index,
        turn_off_progress_bar=True,
    )
    num_parameter = evaluator.num_parameter
    evaluator.evaluate()
    metric_dict = {
        "mean_metrics": evaluator.mean_metrics,
        "median_metrics": evaluator.median_metrics,
        "std_metrics": evaluator.std_metrics,
        "drude_r2": evaluator.drude_r2,
    }
    return num_parameter, metric_dict

def process_one(
    study_path: str,
    study_dir: str,
    results: dict,
    dataloader: torch_geometric.loader.DataLoader,
) -> dict:
    """
    Load and evaluate a single 'study_dir', then insert its entry into the right place in 'results'.
    This version is for the scaling law grid study, where the data and parameter scaling are combined.
    Input:
        study_path:     Path to the root directory containing study result subdirectories
        study_dir:      Name of the study directory in 'study_path'
        results:        Nested dict mapping study types and hyperparameters to dictionaries
        dataloader:     Used to evaluate the best model in the study directory
    Output:
        results:        Nested dict mapping study types and hyperparameters to dictionaries 
                        with 'best_val_loss', 'train_loss', and 'val_loss'
    """
    # path setup and checks
    study_dir_path = os.path.join(study_path, study_dir)
    val_loss_path = os.path.join(study_dir_path, "val_loss.txt")
    best_model_path = os.path.join(study_dir_path, "best_model.pt")
    if not os.path.exists(val_loss_path) or not os.path.exists(best_model_path):
        print(f"Skipping {study_dir_path:s}, probably still running")
        return
    # parse the seed (metadata)
    seed = int(re.search(r"seed(\d+)", study_dir).group(1))
    # load the data from the tensorboard log and validation loss file
    best_val_loss = float(np.loadtxt(val_loss_path))
    tb_log = load_tb_scalars(study_dir_path)
    val_loss = tb_log.get("val/loss", [])
    min_idx = np.argmin(val_loss)
    best_eps_loss = tb_log.get("val/eps", [])[min_idx]
    best_drude_loss = tb_log.get("val/drude", [])[min_idx]
    result_entry = {
        "study_dir": study_dir,
        "seed": seed,
        "val_loss": best_val_loss,
        "eps_loss": best_eps_loss,
        "drude_loss": best_drude_loss,
    }
    # use the "Evaluator" class to evaluate the best model and obtain more detailed metrics
    # (this can take some time...)
    num_parameter, metric_dict = eval_model(
        best_model_path=best_model_path,
        dataloader=dataloader,
    )
    result_entry["num_parameter"] = num_parameter
    result_entry["metric_dict"] = metric_dict
    # put the result entry into the right place in the results dictionary
    num_data = re.search(r"data(\d+)", study_dir).group(1)
    width = re.search(r"width(\d+)", study_dir).group(1)
    if "lmax0" in study_dir:
        results["lmax0"].setdefault(num_data, {}).setdefault(width, []).append(result_entry)
    elif "lmax1" in study_dir:
        results["lmax1"].setdefault(num_data, {}).setdefault(width, []).append(result_entry)
    elif "lmax2" in study_dir:
        results["lmax2"].setdefault(num_data, {}).setdefault(width, []).append(result_entry)
    elif "lmax3" in study_dir:
        results["lmax3"].setdefault(num_data, {}).setdefault(width, []).append(result_entry)
    else:
        raise ValueError(f"Unexpected study_dir format: {study_dir:s}")

# Load the results from the "grid search"

In [ ]:
# directory containing the scaling law study
study_path = "/scratch/magr4985/scaling_law"

# directory to save the results
output_dir = "./scaling_data"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# check the all training runs converged
if os.path.exists(study_path):
    # load all study directories, valdation loss files, and tensorboard logs
    study_dirs = [d for d in os.listdir(study_path) if os.path.isdir(os.path.join(study_path, d))]
    print(f"[INFO] Found {len(study_dirs):d} study directories in {study_path:s}")
    checks = []
    for study_dir in study_dirs:
        val_loss = None
        if os.path.exists(os.path.join(study_path, study_dir, "val_loss.txt")):
            val_loss = float(np.loadtxt(os.path.join(study_path, study_dir, "val_loss.txt")))
        checks.append([study_dir, val_loss, load_tb_scalars(os.path.join(study_path, study_dir))])
    # print the study directories with diverging gradients, i.e, where something went wrong
    print("\n[DIVERGING GRADIENTS CHECK]")
    for study_dir, val_loss, tb_log in checks:
        if np.max(tb_log["train/grad_norm"]) > 1e2:
            print("[DIVERGING GRADIENTS] ", study_dir, np.max(tb_log["train/grad_norm"]))
    # print study directories with training not finished
    print("\n[TRAINING FINISHED CHECK]")
    for study_dir, val_loss, tb_log in checks:
        if len(tb_log.get("val/loss", [])) < 500:
            print("[TRAINING NOT FINISHED] ", study_dir, len(tb_log.get("val/loss", [])))
    # seed-to-seed validation-loss consistency
    print("\n[SEED VARIANCE CHECK]")
    max_seed_variance_thr = 0.05
    _seed_suffix_re = re.compile(r'([_-])seed\d+$')
    _seed_extract_re = re.compile(r'seed(\d+)$')
    def _key_without_seed(name: str) -> str:
        return _seed_suffix_re.sub('', name)
    def _seed_or_inf(name: str) -> int:
        m = _seed_extract_re.search(name)
        return int(m.group(1)) if m else 10**9 # for nice sorting when seed missing
    groups = defaultdict(list)
    for study_dir, val_loss, _ in checks:
        if val_loss is not None and np.isfinite(val_loss):
            groups[_key_without_seed(study_dir)].append((study_dir, float(val_loss)))
    for cfg_key, items in groups.items():
        if len(items) < 2:
            continue # need at least two seeds to compare
        vals = [v for _, v in items]
        sorted_vals = sorted(vals)
        low = sorted_vals[0]
        mid = sorted_vals[len(sorted_vals) // 2]
        high = sorted_vals[-1]
        if low <= 0:
            continue # avoid divide-by-zero/undefined relative difference
        max_rel_spread = max((mid - low) / mid, (high - mid) / mid)
        if max_rel_spread > max_seed_variance_thr:
            print(f"[SEED VARIANCE > 5%] {cfg_key:s}: max(spread)={100*max_rel_spread:.2f}% (low={low:.6g}, mid={mid:.6g} , high={high:.6g})")
            for sd, v in sorted(items, key=lambda p: _seed_or_inf(p[0])):
                print(f"    {sd:s}: val_loss={v:.6g}")

In [ ]:
"""
Incrementally load and evaluate all models in the scaling-law study, caching results to disk.

Evaluating every model can be time-consuming, so this process saves intermediate results
to a JSON file and, on subsequent runs, will only process any newly added models.
If execution is interrupted, you can simply rerun and it will resume from the last saved state.
"""

# path of the JSON file where results are stored
json_path = os.path.join(output_dir, "scaling_results.json")

if os.path.exists(study_path):
    # batch size (speeds up the process at bit)
    batch_size = 256 # adjust this according to your GPU memory
    # path to the validation dataset
    eval_path = "../graph_e3/val.pt"
    # load the data on which we want to evaluate the models
    eval_data = load_torch_data(eval_path)
    dataloader = create_dataloader(
        eval_data, 
        num_data=-1, # use the whole dataset 
        batch_size=batch_size,
        shuffle=False, # do not shuffle the validation set
    )
    print(f"Loaded evaluation data from '{eval_path:s}'", flush=True)
    # load or initialize results
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            results = json.load(f)
        print(f"Loaded existing results ({len(extract_processed_dirs(results)):d} runs)")
    else:
        results = init_empty_results()
        print("Initialized new results store")
    # find what hass already been done
    processed = extract_processed_dirs(results)
    # scan for all study subdirectories
    all_dirs = sorted(d for d in os.listdir(study_path) if os.path.isdir(os.path.join(study_path, d)))
    # only process the new ones
    new_dirs = [d for d in all_dirs if d not in processed]
    save_every = 4  
    total = len(new_dirs)
    if total == 0:
        print("No new studies to process, everything is up to date")
    else:
        print(f"Processing {len(new_dirs)} new studies (saving every {save_every:d})")
        for idx, study_dir in enumerate(new_dirs, start=1):
            process_one(
                study_path=study_path,
                study_dir=study_dir, 
                results=results, 
                dataloader=dataloader,
            )
            # checkpoint every 'save_every' or on the very last one
            if (idx % save_every == 0) or (idx == total):
                with open(json_path, "w") as f:
                    json.dump(results, f, indent=4)
                print(f"    Checkpointed after {idx:d}/{total:d} runs")
        print(f"All {total:d} new runs appended and final JSON saved")
else:
    print(f"All models were processed already!")